# 01 — Pipeline Limpo (2016–2024)

Constrói o corpus rotulado do TCU com **feature única VOTO_LIMPO** (sem vazamento).
Nenhuma célula neste notebook cria features a partir do SUMARIO.

**Saída:**
- `data/interim/acordaos_rotulados.parquet` — corpus 2016–2024 rotulado.
- `data/processed/{train,val,test}.parquet` — splits (temporal por padrão).
- `resultados/metricas_pipeline.json` — sumário do corpus e auditoria de vazamento.


## 1. Setup


In [7]:
import os, sys, subprocess
REPO_DIR = os.environ.get('REPO_DIR', '/content/deep-acordao-tcu2')
REPO_URL = 'https://github.com/bsousa7/deep-acordao-tcu2.git'
BRANCH = os.environ.get('BRANCH', 'claude/deep-acordao-tcu-refactor-yyjfr3')
if not os.path.isdir(os.path.join(REPO_DIR, 'src')):
    subprocess.run(['git', 'clone', REPO_URL, '--branch', BRANCH, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('repo em', REPO_DIR)


repo em /content/deep-acordao-tcu2


In [8]:
%pip -q install pandas pyarrow scikit-learn scipy nltk requests tqdm


## 2. Configuração — escopo temporal 2016–2024


In [9]:
from pathlib import Path
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

ANOS = list(range(2016, 2025))         # 2016..2024 (9 anos)
ANO_VAL = 2023
ANO_TESTE = 2024
RANDOM_STATE = 42

BASE = Path(REPO_DIR)
DATA_RAW = BASE / 'data' / 'raw'
DATA_INTERIM = BASE / 'data' / 'interim'
DATA_PROCESSED = BASE / 'data' / 'processed'
RESULTADOS = BASE / 'resultados'
for d in [DATA_RAW, DATA_INTERIM, DATA_PROCESSED, RESULTADOS / 'figuras']:
    d.mkdir(parents=True, exist_ok=True)
print(f'Anos alvo: {ANOS}')


Anos alvo: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


## 3. CSVs anuais — Google Drive (primário) + download TCU (fallback)

Os CSVs anuais pesam ~200–500 MB cada. No Colab, o mais barato é mantê-los no seu
Google Drive e apenas apontá-los para `data/raw/` via *symlink* — não copia bytes.

Fluxo desta célula:
1. Monta o Google Drive (se ainda não estiver montado).
2. Para cada ano de 2016..2024, procura `acordao-completo-{ano}.csv` em
   `/content/drive/MyDrive/deep-acordao-tcu2/data/raw/` (ajuste `DRIVE_RAW` se
   guardar em outra pasta) e cria um symlink em `data/raw/`.
3. Se algum ano faltar no Drive, tenta baixar do portal TCU como fallback.


In [10]:
from pathlib import Path
import os
from src.aquisicao.baixar_csvs import baixar_todos

# --------------------------------------------------------------
# 1. Monta o Google Drive (opcional — só executa em Colab)
# --------------------------------------------------------------
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
except Exception as _e:
    DRIVE_ROOT = None
    print(f'Google Drive indisponível (fora do Colab?): {_e}')

# --------------------------------------------------------------
# 2. Configure aqui a pasta do Drive onde estão seus CSVs
# --------------------------------------------------------------
# Ajuste se você guardar os CSVs em outro caminho do Drive.
DRIVE_RAW = (DRIVE_ROOT / 'deep-acordao-tcu2' / 'data' / 'raw') if DRIVE_ROOT else None
if DRIVE_RAW is not None:
    print(f'Procurando CSVs em: {DRIVE_RAW}')
    if not DRIVE_RAW.exists():
        print('  (pasta ainda não existe no Drive — nenhum CSV será encontrado por aqui)')

# --------------------------------------------------------------
# 3. Symlink Drive -> data/raw/  (sem duplicar bytes)
# --------------------------------------------------------------
TAM_MIN = 50_000_000

def _apontar_para_drive(ano: int) -> Path | None:
    destino = DATA_RAW / f'acordao-completo-{ano}.csv'
    # Já resolvido?
    if destino.exists() and not destino.is_symlink() and destino.stat().st_size >= TAM_MIN:
        return destino
    if destino.is_symlink() and destino.exists() and destino.stat().st_size >= TAM_MIN:
        return destino
    # Procura no Drive
    if DRIVE_RAW is None:
        return None
    origem = DRIVE_RAW / f'acordao-completo-{ano}.csv'
    if not origem.exists() or origem.stat().st_size < TAM_MIN:
        return None
    # Remove qualquer symlink/arquivo parcial e cria symlink limpo
    if destino.exists() or destino.is_symlink():
        destino.unlink()
    try:
        destino.symlink_to(origem)
    except OSError:
        # Se symlink não for permitido no FS, copia (fallback lento)
        import shutil
        shutil.copy2(origem, destino)
    return destino

presentes, faltantes = {}, []
for ano in ANOS:
    p = _apontar_para_drive(ano)
    if p is not None and p.exists() and p.stat().st_size >= TAM_MIN:
        presentes[ano] = p
        origem = 'Drive' if p.is_symlink() else 'local'
        print(f'  {ano}: {p.stat().st_size/1e6:5.0f} MB  ✓ ({origem})')
    else:
        faltantes.append(ano)
        print(f'  {ano}: ausente')

# --------------------------------------------------------------
# 4. Fallback — baixa do portal TCU apenas o que falta no Drive
# --------------------------------------------------------------
if faltantes:
    print(f'\nBaixando do TCU (não encontrados no Drive): {faltantes}')
    baixar_todos(faltantes, data_dir=DATA_RAW)

arquivos = {a: DATA_RAW / f'acordao-completo-{a}.csv'
            for a in ANOS
            if (DATA_RAW / f'acordao-completo-{a}.csv').exists()}
print(f'\nAnos disponíveis: {sorted(arquivos)}')
assert arquivos, 'Nenhum CSV disponível em data/raw/ nem no Google Drive.'


Procurando CSVs em: /content/drive/MyDrive/deep-acordao-tcu2/data/raw
  (pasta ainda não existe no Drive — nenhum CSV será encontrado por aqui)
  2016:   347 MB  ✓ (local)
  2017:   336 MB  ✓ (local)
  2018:   380 MB  ✓ (local)
  2019:   377 MB  ✓ (local)
  2020:   513 MB  ✓ (local)
  2021:   524 MB  ✓ (local)
  2022:   425 MB  ✓ (local)
  2023:   445 MB  ✓ (local)
  2024:   421 MB  ✓ (local)

Anos disponíveis: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


## 4. Filtro temático + rotulagem


In [11]:
from src.preprocessamento.filtrar_tematico import combinar_anos, salvar_parquet

df = combinar_anos(sorted(arquivos.keys()), data_dir=DATA_RAW, apenas_tema=True)
print('n =', len(df))
print(df['LABEL'].value_counts().to_string())
print()
print('Distribuição por ano:')
print(df.groupby(['ANO', 'LABEL']).size().unstack(fill_value=0).to_string())


n = 3644
LABEL
Irregular               3310
Regular com Ressalva     241
Regular                   93

Distribuição por ano:
LABEL  Irregular  Regular  Regular com Ressalva
ANO                                            
2016         255        9                    19
2017         411        8                    16
2018         363        8                    17
2019         326        4                    27
2020         440        9                    28
2021         457       21                    36
2022         415       14                    26
2023         300        8                    34
2024         343       12                    38


## 5. Construção da feature única — `VOTO_LIMPO`

Remove o dispositivo do VOTO e mascara termos de veredito residuais. Nenhuma feature
é derivada do SUMARIO. O gate de auditoria exige fração de vazamento = 0.


In [12]:
from src.preprocessamento.anti_vazamento import construir_feature_voto, auditar_vazamento, contem_veredito

feats = df['VOTO'].apply(construir_feature_voto)
df['VOTO_LIMPO'] = feats.apply(lambda f: f.texto)
df['dispositivo_encontrado'] = feats.apply(lambda f: f.dispositivo_encontrado)

print(f'Dispositivo localizado em {100*df["dispositivo_encontrado"].mean():.1f}% dos votos')

# Auditoria comparativa (SUMARIO só para diagnóstico — jamais será usado como feature)
aud_sumario = auditar_vazamento(df, 'SUMARIO', verbose=True)
aud_voto_limpo = auditar_vazamento(df, 'VOTO_LIMPO', verbose=True)
assert aud_voto_limpo['gate_passou'], 'Gate de vazamento FALHOU em VOTO_LIMPO — investigar padrões de dispositivo.'
print('GATE VOTO_LIMPO: OK — feature honesta para treino.')


Dispositivo localizado em 26.0% dos votos
[ANTI-VAZAMENTO] Campo: SUMARIO | Gate: FALHOU
  Total:    3,644
  Vazados:  3,574  (98.08%)
[ANTI-VAZAMENTO] Campo: VOTO_LIMPO | Gate: PASSOU
  Total:    3,644
  Vazados:  0  (0.00%)
GATE VOTO_LIMPO: OK — feature honesta para treino.


## 6. Filtro de tamanho mínimo e persistência


In [13]:
N_MIN_CHARS = 200
n_antes = len(df)
df = df[df['VOTO_LIMPO'].str.len() >= N_MIN_CHARS].reset_index(drop=True)
print(f'Após filtro >= {N_MIN_CHARS} chars: {len(df)} (removidos {n_antes - len(df)})')

salvar_parquet(df, DATA_INTERIM / 'acordaos_rotulados.parquet')


Após filtro >= 200 chars: 3644 (removidos 0)


## 7. Split — temporal (padrão) ou estratificado (fallback)

**Temporal (recomendado):** treino ≤ 2022, val = 2023, teste = 2024. Espelha o uso real
e evita 'ver o futuro'.

**Estratificado 70/15/15:** fallback para corpora pequenos por classe.


In [14]:
from collections import Counter
from src.preprocessamento.split_temporal import (
    dividir_temporal_por_ano, dividir_estratificado, salvar_splits,
)

# Verifica cobertura mínima por classe/ano antes de decidir
cobertura = df.groupby(['ANO', 'LABEL']).size().unstack(fill_value=0)
print(cobertura)

usar_temporal = ANO_VAL in df['ANO'].unique() and ANO_TESTE in df['ANO'].unique()
if usar_temporal:
    train_df, val_df, test_df = dividir_temporal_por_ano(df, ano_teste=ANO_TESTE, ano_val=ANO_VAL)
    tipo_split = 'temporal'
else:
    train_df, val_df, test_df = dividir_estratificado(df)
    tipo_split = 'estratificado'

salvar_splits(train_df, val_df, test_df, DATA_PROCESSED)
print(f'\nSplit tipo = {tipo_split}')
print(f'treino={len(train_df)} val={len(val_df)} teste={len(test_df)}')


LABEL  Irregular  Regular  Regular com Ressalva
ANO                                            
2016         255        9                    19
2017         411        8                    16
2018         363        8                    17
2019         326        4                    27
2020         440        9                    28
2021         457       21                    36
2022         415       14                    26
2023         300        8                    34
2024         343       12                    38

Split tipo = temporal
treino=2909 val=342 teste=393


## 8. Sumário e persistência


In [15]:
from src.avaliacao.metricas import salvar_json

sumario = {
    'escopo': {
        'anos': ANOS,
        'anos_disponiveis': sorted(arquivos.keys()),
        'ano_val': ANO_VAL,
        'ano_teste': ANO_TESTE,
        'tipo_split': tipo_split,
    },
    'corpus': {
        'total_apos_filtro_tematico_e_tamanho': int(len(df)),
        'distribuicao_global': df['LABEL'].value_counts().to_dict(),
        'distribuicao_por_ano': df.groupby(['ANO','LABEL']).size().unstack(fill_value=0).to_dict(),
        'dispositivo_localizado_frac': float(df['dispositivo_encontrado'].mean()),
    },
    'auditoria_vazamento': {
        'SUMARIO_referencia': aud_sumario,
        'VOTO_LIMPO_feature': aud_voto_limpo,
    },
    'splits': {
        'treino': {'n': int(len(train_df)), 'dist': train_df['LABEL'].value_counts().to_dict()},
        'val':    {'n': int(len(val_df)),   'dist': val_df['LABEL'].value_counts().to_dict()},
        'teste':  {'n': int(len(test_df)),  'dist': test_df['LABEL'].value_counts().to_dict()},
    },
}
salvar_json(sumario, RESULTADOS / 'metricas_pipeline.json')
print('OK — dados prontos para 02_baseline_ponderado.ipynb e 03_textcnn_ponderado.ipynb.')


OK — dados prontos para 02_baseline_ponderado.ipynb e 03_textcnn_ponderado.ipynb.


In [17]:
from pathlib import Path
import shutil

REPO_DIR_LOCAL = Path(globals().get('REPO_DIR', '/content/deep-acordao-tcu2'))

from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive') / 'deep-acordao-tcu2'

pares = [
    (REPO_DIR_LOCAL / 'data' / 'interim', DRIVE_ROOT / 'data' / 'interim'),
    (REPO_DIR_LOCAL / 'data' / 'processed', DRIVE_ROOT / 'data' / 'processed'),
    (REPO_DIR_LOCAL / 'resultados', DRIVE_ROOT / 'resultados'),
]

for origem, destino in pares:
    if not origem.exists():
        print(f'AVISO: {origem} não existe — pulando.')
        continue
    destino.mkdir(parents=True, exist_ok=True)
    for f in origem.rglob('*'):
        if f.is_file():
            alvo = destino / f.relative_to(origem)
            alvo.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, alvo)
    print(f'Copiado: {origem} -> {destino}')

print('\nConferindo o que ficou no Drive:')
for _, destino in pares:
    for f in sorted(destino.rglob('*')):
        if f.is_file():
            print(' ', f, f'{f.stat().st_size/1e6:.2f} MB')

Copiado: /content/deep-acordao-tcu2/data/interim -> /content/drive/MyDrive/deep-acordao-tcu2/data/interim
Copiado: /content/deep-acordao-tcu2/data/processed -> /content/drive/MyDrive/deep-acordao-tcu2/data/processed
Copiado: /content/deep-acordao-tcu2/resultados -> /content/drive/MyDrive/deep-acordao-tcu2/resultados

Conferindo o que ficou no Drive:
  /content/drive/MyDrive/deep-acordao-tcu2/data/interim/.gitkeep 0.00 MB
  /content/drive/MyDrive/deep-acordao-tcu2/data/interim/acordaos_rotulados.parquet 40.83 MB
  /content/drive/MyDrive/deep-acordao-tcu2/data/processed/.gitkeep 0.00 MB
  /content/drive/MyDrive/deep-acordao-tcu2/data/processed/test.parquet 4.86 MB
  /content/drive/MyDrive/deep-acordao-tcu2/data/processed/train.parquet 32.03 MB
  /content/drive/MyDrive/deep-acordao-tcu2/data/processed/val.parquet 3.98 MB
  /content/drive/MyDrive/deep-acordao-tcu2/resultados/figuras/.gitkeep 0.00 MB
  /content/drive/MyDrive/deep-acordao-tcu2/resultados/metricas_pipeline.json 0.00 MB
